In [21]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.1")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")

assert API_KEY, "OPENAI_API_KEY not found - add it to your .env file."

In [22]:
from openai import OpenAI
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [23]:
def ask(messages):
    """Send a message list to the model and return the assistant reply."""
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## Simple prompt example

In [24]:
simple = [
    {"role": "system", "content": "Translate the user text into Slovenian:"},
    {"role": "user", "content": "She bought an Apple computer"},
]

print(ask(simple))

Ona je kupila računalnik znamke Apple.


## Detailed prompt example

In [25]:
detailed = [
    {
        "role": "system",
        "content": (
            "You are a translation assistant. Translate the user text into Slovenian.\n\n"
            "Additional instructions:\n"
            "- Keep sentence boundaries.\n"
            "- Do not translate product names.\n\n"
            "Return only the translation, no additional explanation."
        ),
    },
    {"role": "user", "content": "She bought an Apple computer."},
]
print(ask(detailed))

Ona je kupila Apple računalnik.


## Multi-turn conversation prompt example

In [26]:
multi_turn = [
    {"role": "system", "content": "Translate the user text into Slovenian"},
    {"role": "user", "content": "She bought an Apple computer"},
    {"role": "assistant", "content": "Kupila je nov računalnik Jabolko"},
    {
        "role": "user",
        "content": (
            "That doesn't seem right. Apple is a product name in this case. "
            "Try again."
        ),
    },
]
print(ask(multi_turn))

Kupila je računalnik znamke Apple.


## In-context learning v1: enriching the system prompt

In [27]:
inline_icl = [
    {
        "role": "system",
        "content": (
            "Translate the user text into Slovenian.\n"
            "Examples:\n"
            "EN: Save changes.\n"
            "SL: Shrani spremembe.\n"
            "EN: Couldn't connect.\n"
            "SL: Povezava ni uspela."
        ),
    },
    {"role": "user", "content": "User text: The file was deleted."},
]
print(ask(inline_icl))

Datoteka je bila izbrisana.


## In-context learning v2: multi-turn conversation

In [28]:
icl_as_turns = [
    {"role": "system", "content": "Translate the user text into Slovenian."},
    {"role": "user", "content": "EN: Save changes."},
    {"role": "assistant", "content": "SL: Shrani spremembe."},
    {"role": "user", "content": "EN: Couldn't connect."},
    {"role": "assistant", "content": "SL: Povezava ni uspela."},
    {"role": "user", "content": "EN: The file was deleted."},
]
print(ask(icl_as_turns))

SL: Datoteka je bila izbrisana.


## Structured output prediction

In [29]:
cot_prompt = [
    {
        "role": "system",
        "content": (
            "Translate the user text into Slovenian.\n"
            "Before translating, list any ambiguities in the text and how "
            "you resolve them. Then generate the translation."
        ),
    },
    {"role": "user", "content": "User text: She bought a new Apple computer."},
]


In [30]:
from pydantic import BaseModel
class Translation(BaseModel):
    ambiguities: list[str]
    translation: str

# Schema for the API. strict mode requires additionalProperties: false.
schema = Translation.model_json_schema()
schema["additionalProperties"] = False

In [31]:
response = client.chat.completions.create(
    model=MODEL,
    messages=cot_prompt,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "translation",
            "schema": schema,
            "strict": True,
        },
    },
)

# Content is a JSON string; validate it back into the pydantic model.
result = Translation.model_validate_json(response.choices[0].message.content)

import json
print(json.loads(result.model_dump_json()))

{'ambiguities': ["Does 'Apple' refer to the brand (Apple) rather than the company in a corporate sense; resolved as referring to the brand and translated as 'znamke Apple'.", "Does 'new' mean brand-new or merely not previously owned by the subject; resolved as brand-new, using 'nov' to modify 'računalnik'.", "Should the Slovenian translation include the subject 'She' explicitly or omit it; resolved by using natural Slovenian order with the subject omitted: 'Kupila je ...' instead of 'Ona je kupila ...'."], 'translation': 'Kupila je nov računalnik znamke Apple.'}
